# 02: Sanity-checking a model before trusting it

A metabolic model is a hand-curated database and it has bugs. Before you believe "delete gene X for 2x yield", check the model in cases where we already know the right answer. This notebook runs four kinds of checks:

1. **Physics:** no growth or ATP from nothing.
2. **Chemistry:** reactions balance their atoms.
3. **Biology:** predicted essential genes match lab experiments.
4. **MEMOTE:** the community's standard automated test suite for models.

In [1]:
import warnings
from sugar2compound import free_energy_checks, gene_essentiality_check, load_model, mass_balance_problems, set_glucose_medium

warnings.filterwarnings("ignore", message="Solver status is 'infeasible'")
core = load_model("e_coli_core")
big = load_model("iML1515", fallback="iJO1366")
for m in (core, big):
    set_glucose_medium(m, aerobic=True)
print(core.id, big.id)

Could not fetch iML1515 from BiGG (The connection to the BiGG Models repository failed.); using iJO1366 instead.


e_coli_core iJO1366


## 1. Free lunches

If a model can grow, or regenerate ATP, with every input closed, it contains a loop that violates thermodynamics (like a perpetual motion machine). Such a model will over-predict every yield. All rows should say `ok = True`.

In [2]:
free_energy_checks(core)

,check,value,ok
0,growth with no uptake at all,0.0,True
1,ATP made from nothing,0.0,True
2,growth with no carbon source,0.0,True


In [3]:
free_energy_checks(big)

,check,value,ok
0,growth with no uptake at all,0.0,True
1,ATP made from nothing,0.0,True
2,growth with no carbon source,0.0,True


## 2. Mass and charge balance

A reaction like `A -> B` must have the same atoms on both sides. Imbalanced reactions can create carbon from nothing. Some imbalances are deliberate (pseudo-reactions that lump many steps), so read the list rather than expecting it empty.

In [4]:
mass_balance_problems(core)

,reaction,name,imbalance


In [5]:
mass_balance_problems(big)

,reaction,name,imbalance
0,BIOMASS_Ec_iJO1366_WT_53p95M,E. coli biomass objective function (iJO1366) -...,"{'charge': 0.482940000000057, 'C': -40.8584139..."


## 3. Gene essentiality vs the lab

A gene is **essential** if a cell missing it can't grow. The [Keio collection](https://doi.org/10.1038/msb4100050) tried to delete every *E. coli* gene one at a time, so we know the answer for most genes. `gene_essentiality_check` simulates deleting each gene in a small reference list and compares.

On a real project you'd run `cobra.flux_analysis.single_gene_deletion` on *every* gene and compute accuracy against the full Keio data set; this small list is meant to be readable.

In [6]:
gene_essentiality_check(core)

,gene,description,known_essential,predicted_growth,predicted_essential,verdict
0,b1779,gapA: glyceraldehyde-3-phosphate dehydrogenase...,True,0.000000e+00,True,agree
1,b2926,pgk: phosphoglycerate kinase (glycolysis),True,-1.819776e-16,True,agree
2,b2779,eno: enolase (glycolysis),True,0.000000e+00,True,agree
3,b0720,gltA: citrate synthase (no glutamate without it),True,0.000000e+00,True,agree
4,b3956,ppc: PEP carboxylase (refills the TCA cycle on...,True,8.707448e-01,False,MISMATCH
5,b0004,thrC: threonine synthase (makes the amino acid...,True,NaN,None,not in model
6,b3960,argH: argininosuccinate lyase (makes arginine),True,NaN,None,not in model
7,b4177,purA: adenylosuccinate synthase (makes AMP for...,True,NaN,None,not in model
8,b1281,pyrF: OMP decarboxylase (makes pyrimidines for...,True,NaN,None,not in model
9,b4025,pgi: glucose-6-phosphate isomerase (pentose ph...,False,8.631596e-01,False,agree


The core model can't say anything about amino acid or nucleotide genes (they're "not in model"), which is one reason to use the genome-scale model for real work.

In [7]:
gene_essentiality_check(big)

,gene,description,known_essential,predicted_growth,predicted_essential,verdict
0,b1779,gapA: glyceraldehyde-3-phosphate dehydrogenase...,True,8.560605e-01,False,MISMATCH
1,b2926,pgk: phosphoglycerate kinase (glycolysis),True,8.560605e-01,False,MISMATCH
2,b2779,eno: enolase (glycolysis),True,9.049382e-01,False,MISMATCH
3,b0720,gltA: citrate synthase (no glutamate without it),True,5.463499e-31,True,agree
4,b3956,ppc: PEP carboxylase (refills the TCA cycle on...,True,9.793665e-01,False,MISMATCH
5,b0004,thrC: threonine synthase (makes the amino acid...,True,-2.248885e-16,True,agree
6,b3960,argH: argininosuccinate lyase (makes arginine),True,4.709935e-16,True,agree
7,b4177,purA: adenylosuccinate synthase (makes AMP for...,True,-1.173013e-14,True,agree
8,b1281,pyrF: OMP decarboxylase (makes pyrimidines for...,True,0.000000e+00,True,agree
9,b4025,pgi: glucose-6-phosphate isomerase (pentose ph...,False,9.761293e-01,False,agree


**Read the mismatches.** Each one is either a model error or a gap in our understanding, and each is worth five minutes of reading:

* **ppc** is predicted non-essential in both models, but real `ppc` mutants don't grow on glucose minimal medium. The model refills oxaloacetate through the glyoxylate shunt, a route the real cell keeps switched off on glucose by gene regulation, which FBA doesn't model. Notebook 01's knockout search fell for exactly this.
* In the bundled iJO1366, the glycolysis genes **gapA**, **pgk** and **eno** are predicted non-essential too. The model detours around them (through the Entner-Doudoroff pathway, the toxic methylglyoxal bypass, or serine deaminase) at a speed no real cell can manage. The small core model doesn't have those detours, so here it happens to be *more* right.

Each of these is recorded as a documented known failure in `tests/benchmarks/cases.py`.

## 4. MEMOTE

[MEMOTE](https://memote.readthedocs.io) runs ~150 standard tests: annotations (are metabolites linked to databases like ChEBI?), consistency (stoichiometry, blocked reactions, dead-end metabolites), biomass, and more. It writes an HTML report.

Many annotation tests fail for the bundled models because they were stripped of cross-references to keep them small. Look at the **consistency** section first: that's where the science-relevant problems show up. The genome-scale report takes several minutes, so it is off by default.

In [8]:
import pathlib, re, subprocess
from cobra.io import write_sbml_model

reports = pathlib.Path("../reports")
reports.mkdir(exist_ok=True)

def memote_report(model):
    sbml = reports / f"{model.id}.xml"
    html = reports / f"{model.id}_memote.html"
    write_sbml_model(model, str(sbml))
    proc = subprocess.run(["memote", "report", "snapshot", "--filename", str(html), str(sbml)],
                          capture_output=True, text=True)
    # The last line of pytest's output summarises passed/failed tests.
    summary = [re.sub(r"\x1b\[[0-9;]*m", "", line) for line in proc.stdout.splitlines() if " passed" in line or " failed" in line]
    print(summary[-1] if summary else proc.stdout[-2000:] + proc.stderr[-2000:])
    print("report:", html)

memote_report(core)
RUN_BIG = False  # set True to also test the genome-scale model (slow)
if RUN_BIG:
    memote_report(big)

============ 64 failed, 68 passed, 15 skipped, 5 warnings in 1.26s =============
report: ../reports/e_coli_core_memote.html


Open the HTML file in a browser. `reports/` is git-ignored, so the reports stay local.